# HW09 Part B :: Hyper Parameter Optimization (Deep Learning)

COSC 6373 -- Adam Nelson-Archer, 2140122

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from sklearn.model_selection import train_test_split
import time

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.16.2


## 1. Load the dataset & 2. Preprocess the data & 3. Split the data
- Load CIFAR-10
- Normalize pixel values to [0, 1]
- Create validation set (10% of training data)

In [5]:
# 1. Load the dataset
(X_train_full, y_train_full), (X_test, y_test) = datasets.cifar10.load_data()

# 2. Preprocess the data
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"Original Training data shape: {X_train_full.shape}")
print(f"Test data shape: {X_test.shape}")

# 3. Split the data (10% validation from training data)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.10, random_state=42
)

print(f"Final Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")

Original Training data shape: (50000, 32, 32, 3)
Test data shape: (10000, 32, 32, 3)
Final Training data shape: (45000, 32, 32, 3)
Validation data shape: (5000, 32, 32, 3)


## 4. Build the model
Helper function to construct a Convolutional Neural Network (CNN) allowing for hyperparameter tuning.
The model includes at least one convolutional layer, one pooling layer, one dense layer, and an output layer.

In [6]:
def build_model(filters=32, kernel_size=(3,3), learning_rate=0.001):
    model = models.Sequential()
    # At least one convolutional layer
    model.add(layers.Conv2D(filters, kernel_size, activation='relu', input_shape=(32, 32, 3)))
    # At least one pooling layer
    model.add(layers.MaxPooling2D((2, 2)))
    # Flattening before dense layers
    model.add(layers.Flatten())
    # At least one dense layer
    model.add(layers.Dense(64, activation='relu'))
    # Output layer for 10 classes
    model.add(layers.Dense(10, activation='softmax'))
    
    # Compile the model
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## 5. Tune hyperparameters & 6. Train the model
Manually test 3 different configurations varying:
1. Number of filters
2. Kernel size
3. Learning rate
4. Batch size
5. Number of epochs

In [7]:
# Dictionary to store results
experiment_results = {}

### Experiment 1 (Baseline)

In [8]:
# Configuration 1
config_1 = {
    'filters': 32,
    'kernel_size': (3, 3),
    'learning_rate': 0.001,
    'batch_size': 64,
    'epochs': 5
}

print("Running Experiment 1...")
model_1 = build_model(filters=config_1['filters'], 
                      kernel_size=config_1['kernel_size'], 
                      learning_rate=config_1['learning_rate'])

start_time = time.time()
history_1 = model_1.fit(X_train, y_train, 
                        epochs=config_1['epochs'], 
                        batch_size=config_1['batch_size'],
                        validation_data=(X_val, y_val),
                        verbose=1)
train_time = time.time() - start_time

val_acc_1 = history_1.history['val_accuracy'][-1]
experiment_results['Exp 1'] = {'config': config_1, 'val_acc': val_acc_1, 'model': model_1, 'time': train_time}
print(f"Experiment 1 Validation Accuracy: {val_acc_1:.4f}")

Running Experiment 1...


C:\Users\PC\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.3325 - loss: 1.8559 - val_accuracy: 0.4706 - val_loss: 1.4825
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.5258 - loss: 1.3489 - val_accuracy: 0.5490 - val_loss: 1.2711
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.5707 - loss: 1.2177 - val_accuracy: 0.5632 - val_loss: 1.2189
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.5996 - loss: 1.1474 - val_accuracy: 0.5866 - val_loss: 1.1691
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.6204 - loss: 1.0783 - val_accuracy: 0.5964 - val_loss: 1.1808
Experiment 1 Validation Accuracy: 0.5964


### Experiment 2
Varying all hyperparameters: More filters, larger kernel size, larger learning rate, larger batch size, fewer epochs.

In [9]:
# Configuration 2
config_2 = {
    'filters': 64,
    'kernel_size': (5, 5),
    'learning_rate': 0.005,
    'batch_size': 128,
    'epochs': 4
}

print("Running Experiment 2...")
model_2 = build_model(filters=config_2['filters'], 
                      kernel_size=config_2['kernel_size'], 
                      learning_rate=config_2['learning_rate'])

start_time = time.time()
history_2 = model_2.fit(X_train, y_train, 
                        epochs=config_2['epochs'], 
                        batch_size=config_2['batch_size'],
                        validation_data=(X_val, y_val),
                        verbose=1)
train_time = time.time() - start_time

val_acc_2 = history_2.history['val_accuracy'][-1]
experiment_results['Exp 2'] = {'config': config_2, 'val_acc': val_acc_2, 'model': model_2, 'time': train_time}
print(f"Experiment 2 Validation Accuracy: {val_acc_2:.4f}")

Running Experiment 2...
Epoch 1/4
352/352 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.2298 - loss: 2.1470 - val_accuracy: 0.4328 - val_loss: 1.6016
Epoch 2/4
352/352 ━━━━━━━━━━━━━━━━━━━━ 12s 33ms/step - accuracy: 0.4456 - loss: 1.5671 - val_accuracy: 0.4524 - val_loss: 1.5166
Epoch 3/4
352/352 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - accuracy: 0.4851 - loss: 1.4602 - val_accuracy: 0.4646 - val_loss: 1.4919
Epoch 4/4
352/352 ━━━━━━━━━━━━━━━━━━━━ 12s 33ms/step - accuracy: 0.5011 - loss: 1.4061 - val_accuracy: 0.4872 - val_loss: 1.4362
Experiment 2 Validation Accuracy: 0.4872


### Experiment 3
Varying all hyperparameters: Fewer filters, smaller learning rate, smaller batch size, more epochs.

In [10]:
# Configuration 3
config_3 = {
    'filters': 16,
    'kernel_size': (3, 3), # Kept 3x3 as standard, varied other things, but prompt says "must vary all". We varied kernel from 3x3 to 5x5 in Exp 2. Technically they are varied *across* experiments. Let's use (4,4) to be strictly changing it from baseline if needed, but the rule is just that across experiments we change it at least once. We will use (4,4) to be totally distinct.
    'learning_rate': 0.0005,
    'batch_size': 32,
    'epochs': 6
}
# Using a (4,4) kernel to ensure it's different
config_3['kernel_size'] = (4, 4)

print("Running Experiment 3...")
model_3 = build_model(filters=config_3['filters'], 
                      kernel_size=config_3['kernel_size'], 
                      learning_rate=config_3['learning_rate'])

start_time = time.time()
history_3 = model_3.fit(X_train, y_train, 
                        epochs=config_3['epochs'], 
                        batch_size=config_3['batch_size'],
                        validation_data=(X_val, y_val),
                        verbose=1)
train_time = time.time() - start_time

val_acc_3 = history_3.history['val_accuracy'][-1]
experiment_results['Exp 3'] = {'config': config_3, 'val_acc': val_acc_3, 'model': model_3, 'time': train_time}
print(f"Experiment 3 Validation Accuracy: {val_acc_3:.4f}")

Running Experiment 3...
Epoch 1/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.3297 - loss: 1.8507 - val_accuracy: 0.4936 - val_loss: 1.4318
Epoch 2/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.5104 - loss: 1.3928 - val_accuracy: 0.5344 - val_loss: 1.3110
Epoch 3/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.5555 - loss: 1.2737 - val_accuracy: 0.5546 - val_loss: 1.2527
Epoch 4/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.5803 - loss: 1.1977 - val_accuracy: 0.5626 - val_loss: 1.2303
Epoch 5/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.5981 - loss: 1.1500 - val_accuracy: 0.5758 - val_loss: 1.1876
Epoch 6/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6172 - loss: 1.0976 - val_accuracy: 0.5866 - val_loss: 1.1601
Experiment 3 Validation Accuracy: 0.5866


## 7. Evaluate performance

In [11]:
print("--- Hyperparameter Configurations and Validation Accuracies ---")
best_exp = None
best_val_acc = 0.0

for exp_name, data in experiment_results.items():
    print(f"{exp_name}:")
    print(f"  Filters: {data['config']['filters']}")
    print(f"  Kernel Size: {data['config']['kernel_size']}")
    print(f"  Learning Rate: {data['config']['learning_rate']}")
    print(f"  Batch Size: {data['config']['batch_size']}")
    print(f"  Epochs: {data['config']['epochs']}")
    print(f"  --> Validation Accuracy: {data['val_acc']:.4f}")
    print(f"  --> Training Time: {data['time']:.2f} seconds")
    print("-" * 50)
    
    if data['val_acc'] > best_val_acc:
        best_val_acc = data['val_acc']
        best_exp = exp_name

print(f"Best Configuration: {best_exp} with Validation Accuracy: {best_val_acc:.4f}")

# Evaluate best model on test set
best_model = experiment_results[best_exp]['model']
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)

print(f"Final Test Accuracy using the best model ({best_exp}): {test_acc:.4f}")

--- Hyperparameter Configurations and Validation Accuracies ---
Exp 1:
  Filters: 32
  Kernel Size: (3, 3)
  Learning Rate: 0.001
  Batch Size: 64
  Epochs: 5
  --> Validation Accuracy: 0.5964
  --> Training Time: 44.94 seconds
--------------------------------------------------
Exp 2:
  Filters: 64
  Kernel Size: (5, 5)
  Learning Rate: 0.005
  Batch Size: 128
  Epochs: 4
  --> Validation Accuracy: 0.4872
  --> Training Time: 50.29 seconds
--------------------------------------------------
Exp 3:
  Filters: 16
  Kernel Size: (4, 4)
  Learning Rate: 0.0005
  Batch Size: 32
  Epochs: 6
  --> Validation Accuracy: 0.5866
  --> Training Time: 44.90 seconds
--------------------------------------------------
Best Configuration: Exp 1 with Validation Accuracy: 0.5964
Final Test Accuracy using the best model (Exp 1): 0.5910


## 8. Reflection

**a. Which hyperparameters had the biggest impact on performance?**

The learning rate appeared to have the biggest impact. In Experiment 2, increasing the learning rate to 0.005 caused a significant drop in validation accuracy (down to 0.4872) compared to Experiments 1 and 3 (which used 0.001 and 0.0005 and hovered around 0.58-0.59). A learning rate that is too high can cause the optimizer to overshoot local minima, resulting in worse performance despite having a more complex model.

**b. Did increasing model complexity always improve results? Why or why not?**

No, increasing model complexity did not improve results. Experiment 2 had the most complex configuration (64 filters, 5x5 kernel size) but achieved the worst validation accuracy (0.4872). Increasing complexity without appropriately tuning other parameters (like lowering the learning rate or increasing epochs) can lead to suboptimal training or overfitting. A larger model requires careful regularization and sufficient training time.

**c. Why is a validation set important in deep learning?**

A validation set acts as an unbiased evaluator during the model tuning phase. In deep learning, models are powerful enough to perfectly memorize the training set. By tracking the validation accuracy across different hyperparameter configurations (like seeing the accuracy drop in Exp 2), we can evaluate how well the model generalizes to new data and pick the best configuration without "leaking" the test set information into our tuning decisions.

**d. What challenges did you encounter when training the CNN?**

The biggest challenge was balancing the trade-off between training time and performance. Each experiment took around 45-50 seconds to train for just a few epochs. Searching for the optimal hyperparameter combination manually is time-consuming and somewhat arbitrary, as changing one parameter (like learning rate) can drastically alter how other parameters (like filter count) behave.